# 05 - The Integer Hull

An integer program asks us to optimize over lattice points inside a polyhedron. Those points form a discrete, usually non-convex set. The **integer hull** reconnects this discrete problem to convex geometry by taking the convex hull of exactly those feasible lattice points.

## Learning objectives

By the end of this notebook, you should be able to:

1. define pure- and mixed-integer hulls;
2. prove the basic containment and optimization properties of the integer hull;
3. distinguish an LP relaxation, its integer-feasible points, and their integer hull;
4. compute a small two-dimensional integer hull by enumeration;
5. define valid inequalities, cuts, facets, and the integrality gap; and
6. explain why practical algorithms approximate the integer hull rather than construct it explicitly.

## Prerequisites

We use LP relaxations and bounds from Notebook 01 and convex hulls, vertices, H-representations, and V-representations from Notebook 04. A **lattice point** is a vector in $\mathbb{Z}^n$, meaning that every coordinate is an integer.


## From an LP relaxation to integer-feasible points

Let

$$
P=\{x\in\mathbb{R}^n\mid Ax\le b\}
$$

be a polyhedron. We call $P$ the **LP relaxation** when the intended model additionally requires $x\in\mathbb{Z}^n$. The integer-feasible set is

$$
S=P\cap\mathbb{Z}^n.
$$

The set $S$ is generally not convex. If $u,v\in S$ are distinct, the segment between them contains points such as $\tfrac12u+\tfrac12v$, whose coordinates may be fractional.

The **integer hull** of $P$ is

$$
\boxed{P_I=\operatorname{conv}(P\cap\mathbb{Z}^n).}
$$

It is the smallest convex set containing every integer-feasible point. Because $P$ is convex and already contains those points,

$$
P_I\subseteq P.
$$

No integer-feasible point is lost or gained:

$$
P_I\cap\mathbb{Z}^n=P\cap\mathbb{Z}^n.
$$

The inclusion from right to left follows from the definition. For the other direction, any integer point in $P_I$ also lies in $P$, so it already belongs to $P\cap\mathbb{Z}^n$.


## Why linear optimization over the integer hull is equivalent

Let $c\in\mathbb{R}^n$ and suppose the maximum is finite and attained. Every $x\in P_I$ is a convex combination of integer-feasible points $x^1,\ldots,x^q$:

$$
x=\sum_{i=1}^q\lambda_i x^i,
\qquad
\lambda_i\ge0,
\qquad
\sum_i\lambda_i=1.
$$

Linearity gives

$$
c^\top x
=\sum_i\lambda_i c^\top x^i
\le\max_i c^\top x^i.
$$

Therefore, taking convex combinations cannot improve beyond the best generating point, and

$$
\boxed{
\max\{c^\top x\mid x\in P\cap\mathbb{Z}^n\}
=
\max\{c^\top x\mid x\in P_I\}.
}
$$

This equality is the main reason to study $P_I$. If a compact H-representation of the integer hull were available, the integer program could be solved as an LP. The difficulty has moved from the objective to describing the right polyhedron.


## Mixed-integer hulls

In a mixed-integer model, only coordinates in an index set $I$ must be integral. If, after reordering, the first $p$ coordinates are integral, the feasible set is

$$
P\cap\left(\mathbb{Z}^p\times\mathbb{R}^{n-p}\right),
$$

and the **mixed-integer hull** is

$$
P_{I,p}
=
\operatorname{conv}\!\left(
P\cap(\mathbb{Z}^p\times\mathbb{R}^{n-p})
\right).
$$

Two endpoints clarify the notation:

- $p=0$: no variable is required to be integral, so $P_{I,0}=P$;
- $p=n$: all variables are integral, so $P_{I,n}=P_I$.

Unlike a pure-integer feasible set, a mixed-integer feasible set is not discrete in every coordinate: continuous variables can vary along segments. The convex-hull idea is nevertheless the same. Our visual example uses the pure-integer case.


## A two-dimensional example

Consider the rational polytope

$$
P=\left\{x\in\mathbb{R}^2\ \middle|\
\begin{aligned}
x_1&\ge0,\\
x_2&\ge0,\\
2x_1+x_2&\le\frac{17}{2},\\
x_1+2x_2&\le\frac{17}{2}
\end{aligned}
\right\}.
$$

A polyhedron is **rational** if it has an H-representation with rational coefficients. Rational data matter because a finite input file normally stores rational numbers.

The vertices of $P$ are

$$
(0,0),\quad
\left(\frac{17}{4},0\right),\quad
\left(\frac{17}{6},\frac{17}{6}\right),\quad
\left(0,\frac{17}{4}\right).
$$

Three vertices are fractional. They are feasible for the LP relaxation but not for the integer program. The integer-feasible points are the black lattice points inside $P$, and their convex hull has vertices

$$
(0,0),\ (4,0),\ (3,2),\ (2,3),\ (0,4).
$$


### Computing a small integer hull by enumeration

For a bounded two-dimensional polytope, a direct educational algorithm is:

1. compute the vertices of $P$;
2. use their coordinate minima and maxima to obtain a finite integer bounding box;
3. enumerate lattice points in that box;
4. retain the points satisfying $Ax\le b$; and
5. compute the ordinary convex hull of the retained points.

The first helper performs steps 1–4.

```{literalinclude} ../src/polyhedra/integer_hull.py
:language: python
:linenos:
:start-after: "# --- INTEGER_HULL:lattice_points:start"
:end-before: "# --- INTEGER_HULL:lattice_points:end"
```

The second helper performs step 5 and handles lower-dimensional results such as a single point or a line segment.

```{literalinclude} ../src/polyhedra/integer_hull.py
:language: python
:linenos:
:start-after: "# --- INTEGER_HULL:vertices:start"
:end-before: "# --- INTEGER_HULL:vertices:end"
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src import (
    hrep_to_vertices_2d,
    integer_hull_vertices_2d,
    lattice_points_in_polytope_2d,
)

A = np.array(
    [
        [-1.0, 0.0],       # x1 >= 0
        [0.0, -1.0],       # x2 >= 0
        [2.0, 1.0],        # 2*x1 + x2 <= 17/2
        [1.0, 2.0],        # x1 + 2*x2 <= 17/2
    ]
)
b = np.array([0.0, 0.0, 8.5, 8.5]).reshape(-1, 1)

polytope_vertices = hrep_to_vertices_2d(A, b)
lattice_points = lattice_points_in_polytope_2d(A, b)
integer_hull_vertices = integer_hull_vertices_2d(A, b)

print(f"Number of integer feasible points: {len(lattice_points)}")
print("Integer-hull vertices:")
print(integer_hull_vertices)

The next plot distinguishes three sets:

- blue: the continuous relaxation $P$;
- black points: the discrete feasible set $P\cap\mathbb{Z}^2$;
- green: their convex hull $P_I$.

The green set is smaller than the blue set, but both contain exactly the same feasible lattice points.


In [ ]:
def close_polygon(vertices):
    return np.vstack([vertices, vertices[0]])

P_closed = close_polygon(polytope_vertices)
PI_closed = close_polygon(integer_hull_vertices)

fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(P_closed[:, 0], P_closed[:, 1], color="tab:blue", alpha=0.15, label="$P$ (LP relaxation)")
ax.plot(P_closed[:, 0], P_closed[:, 1], color="tab:blue")
ax.fill(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", alpha=0.25, label="$P_I$ (integer hull)")
ax.plot(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", linewidth=2)
ax.scatter(lattice_points[:, 0], lattice_points[:, 1], color="black", s=25, zorder=3, label=r"$P\cap\mathbb{Z}^2$")
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("LP relaxation, integer points, and integer hull")
ax.grid(True)
ax.legend()
plt.show()

### Why enumeration does not scale

If a bounding box contains $N_i$ possible integer values for coordinate $i$, enumeration checks roughly

$$
N_1N_2\cdots N_n
$$

points. This product grows exponentially with dimension, and large coordinate ranges cause trouble even in modest dimension. Unbounded polyhedra need additional arguments because they do not have a finite bounding box.

The helper is therefore appropriate for visualization, tiny examples, and test oracles—not as a general integer-hull algorithm. The theoretical fact that the integer hull of a rational polyhedron is itself a rational polyhedron does **not** imply that its inequalities are easy to find or few in number.


## Comparing the LP and integer optima

Maximize

$$
c^\top x=x_1+x_2
$$

in the example. The two slanted constraints are tight at the LP optimum, so

$$
x^{LP}=\left(\frac{17}{6},\frac{17}{6}\right),
\qquad
z_{LP}=\frac{17}{3}\approx5.67.
$$

This point is fractional. The best integer solutions are $(3,2)$ and $(2,3)$, both with value

$$
z_{IP}=5.
$$

For a maximization problem,

$$
z_{IP}=z_{P_I}\le z_{LP}.
$$

The **additive integrality gap** for this instance and objective is $z_{LP}-z_{IP}=2/3$. A gap depends on the formulation and the objective; it is not a property of the feasible integer points alone. Relative-gap conventions also exist and require care when objective values can be zero or have different signs.


In [ ]:
from scipy.optimize import linprog

c = np.array([1.0, 1.0])
lp_result = linprog(-c, A_ub=A, b_ub=b.ravel(), bounds=[(None, None)] * 2, method="highs")
integer_values = lattice_points @ c
best_integer_value = integer_values.max()
best_integer_points = lattice_points[np.isclose(integer_values, best_integer_value)]

print("LP optimum:", lp_result.x, "with value", -lp_result.fun)
print("Best integer points:", best_integer_points.tolist())
print("Integer optimum value:", best_integer_value)
print("Additive relaxation gap:", -lp_result.fun - best_integer_value)

### Valid inequalities, cuts, and facets

An inequality $\alpha^\top x\le\beta$ is **valid** for the integer-feasible set if every point in $P\cap\mathbb{Z}^n$ satisfies it. Adding a valid inequality to the LP relaxation does not remove any admissible integer solution.

For the example, add the two slanted constraints:

$$
3x_1+3x_2\le17.
$$

At integer points, $x_1+x_2$ is an integer. Therefore,

$$
3(x_1+x_2)\le17
\quad\Longrightarrow\quad
x_1+x_2\le\left\lfloor\frac{17}{3}\right\rfloor=5.
$$

Thus

$$
\boxed{x_1+x_2\le5}
$$

is valid for all integer-feasible points, although it is not valid for the entire LP relaxation. It removes the fractional optimum because $17/3>5$. A valid inequality used to remove a current fractional relaxation solution is called a **cut** or **cutting plane**.

The equality $x_1+x_2=5$ supports the edge between $(3,2)$ and $(2,3)$ of the two-dimensional integer hull. Hence this inequality defines a **facet** of $P_I$. Facet-defining inequalities are geometrically strong, but a useful cut need not define a facet.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(P_closed[:, 0], P_closed[:, 1], color="tab:blue", alpha=0.15, label="$P$")
ax.plot(P_closed[:, 0], P_closed[:, 1], color="tab:blue")
ax.fill(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", alpha=0.25, label="$P_I$")
ax.plot(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", linewidth=2)
ax.scatter(*lp_result.x, marker="x", s=100, linewidths=2, color="tab:red", label="LP optimum")
ax.scatter(best_integer_points[:, 0], best_integer_points[:, 1], marker="*", s=130, color="black", label="integer optima")
cut_x = np.linspace(0, 5, 100)
ax.plot(cut_x, 5 - cut_x, linestyle="--", color="tab:orange", label="$x_1+x_2=5$")
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("A valid inequality cuts off the fractional LP optimum")
ax.grid(True)
ax.legend()
plt.show()

## Integral polyhedra

A rational polyhedron $P$ is **integral** if

$$
P=P_I.
$$

For a rational polytope, this is equivalent to every vertex being integral. If an LP over an integral polytope has an optimum, at least one optimal vertex is integral, so the LP already solves the corresponding integer problem.

The example polytope is not integral because it has fractional vertices. Its integer hull is integral by construction. Notebook 07 introduces total unimodularity, an important structural condition under which integral right-hand sides imply an integral polyhedron.

Do not confuse an **integral polyhedron** with a set containing only integer points. An integral polyhedron is convex and therefore contains fractional points on segments between its integer vertices; the claim is that it is the convex hull of its integer points.


## How algorithms use the integer-hull idea

Explicitly constructing all of $P_I$ is usually impractical. Modern integer-optimization algorithms reveal only enough of its structure to prove optimality:

- **Cutting-plane methods** add valid inequalities and move the relaxation toward $P_I$.
- **Branch-and-bound** divides the feasible region into smaller subproblems and uses bounds to discard regions that cannot improve the best known integer solution.
- **Branch-and-cut** combines both approaches.

At any stage, the solver works with an outer approximation

$$
P_I\subseteq Q\subseteq P.
$$

A tighter $Q$ can improve bounds and reduce search, but stronger inequalities may be expensive to discover or may make LPs harder to solve. Practical optimization balances relaxation strength against computational cost.


## Check your understanding

1. Why is $P_I\subseteq P$?
2. Why does maximizing a linear function over $P_I$ give the same value as maximizing it over $P\cap\mathbb{Z}^n$?
3. Does an integral polytope contain only integer points?
4. Derive $x_1+x_2\le5$ from the two slanted inequalities and integrality.
5. In a maximization problem, which is larger: $z_{LP}$ or $z_{IP}$?

Brief answers: (1) $P$ is convex and contains all generators of $P_I$; (2) a convex combination's objective is a weighted average of generating-point objectives; (3) no, its edges and interior generally contain fractional points; (4) add to obtain $3(x_1+x_2)\le17$, divide, and round down the integral left side; (5) $z_{LP}\ge z_{IP}$.


## Takeaways and further reading

- The integer hull is $P_I=\operatorname{conv}(P\cap\mathbb{Z}^n)$.
- It is convex, lies inside the LP relaxation, and preserves exactly the feasible lattice points.
- Linear optimization over $P_I$ is equivalent to optimization over the integer-feasible set.
- Fractional LP optima can create an integrality gap.
- Valid inequalities tighten a relaxation without excluding integer-feasible points; cuts exclude a current fractional solution.
- Rational integer hulls are polyhedral, but their complete descriptions can be difficult or enormous.

For broader polyhedral integer-programming theory, see the freely available [Integer Programming and Combinatorial Optimization lecture notes by Gérard Cornuéjols](https://integer.tepper.cmu.edu/). The [SCIP Optimization Suite documentation](https://www.scipopt.org/doc/html/) provides a practical view of how modern solvers combine relaxations, cuts, and search.


## Next

The next notebook explains branch-and-bound from the ground up. It defines a subproblem, a node, a branch, and a search tree before using LP bounds and integer incumbents to certify a global optimum.

➡️ [06 - Branch and Bound](06-Branch-And-Bound.ipynb)
